# CALE Experiment Visualization

This notebook visualizes the JSON report produced by `experiment.py`. It now supports both:

1. **supervised settings** with response-level expert labels, where `accuracy` and `macro_f1` are available;
2. **FEVER-style claim datasets** without response-level expert labels, where evaluator comparison should focus on prediction distributions, construct subscores, and stress robustness.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("default")


## 1. Load Results

Set `RESULTS_PATH` to the JSON file generated by `experiment.py`.


In [ ]:
RESULTS_PATH = Path("results.json")
OUTDIR = Path("figures")
OUTDIR.mkdir(parents=True, exist_ok=True)

with RESULTS_PATH.open("r", encoding="utf-8") as f:
    report = json.load(f)

report.keys()


## 2. Metrics Table

If you are using FEVER without response-level expert labels, `accuracy`, `macro_f1`, and `checklist_f1` may be null. In that case, the later sections on prediction distributions and stress robustness become more important.


In [ ]:
metrics_df = (
    pd.DataFrame(report["metrics"])
    .T
    .reset_index()
    .rename(columns={"index": "variant"})
)
metrics_df


## 3. Plot Only Available Metrics

This cell skips metrics that are entirely null.


In [ ]:
def plot_metric(metric_name, ylim=(0, 1)):
    if metric_name not in metrics_df.columns:
        print(f"Missing metric: {metric_name}")
        return
    df = metrics_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"Skipping {metric_name}: no non-null values.")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"metric_{metric_name}.png", dpi=200)
    plt.show()

for metric in [
    "accuracy",
    "macro_f1",
    "checklist_f1",
    "mean_uncertainty",
    "misinformation_detection_rate",
    "framing_resistance_rate",
    "overclaim_rate_on_nei",
]:
    plot_metric(metric, ylim=(0, 1))


## 4. Prediction Table

This table is the most useful view when gold response labels are not available.


In [ ]:
pred_df = pd.DataFrame(report.get("predictions", []))
pred_df.head()


## 5. Prediction Label Distribution by Variant

This shows how often each evaluator variant predicts `corrected`, `partially_corrected`, `uncertain`, or `not_corrected`.


In [ ]:
if not pred_df.empty:
    label_counts = (
        pred_df.groupby(["variant", "label"]).size()
        .reset_index(name="count")
    )
    label_counts
else:
    print("No predictions found.")


In [ ]:
if not pred_df.empty:
    pivot = label_counts.pivot(index="variant", columns="label", values="count").fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0)

    ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 5))
    ax.set_title("Prediction label distribution by evaluator variant")
    ax.set_ylabel("proportion")
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    plt.tight_layout()
    plt.savefig(OUTDIR / "prediction_label_distribution.png", dpi=200)
    plt.show()


## 6. Score Distribution by Variant

This helps compare whether CALE and direct judges produce systematically different score ranges.


In [ ]:
if not pred_df.empty and "score" in pred_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    pred_df.boxplot(column="score", by="variant", ax=ax, grid=False, rot=30)
    ax.set_title("Score distribution by evaluator variant")
    ax.set_ylabel("score")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig(OUTDIR / "score_distribution_by_variant.png", dpi=200)
    plt.show()


## 7. Stress Summary

These plots compare robustness under perturbations. Lower invariance score shift and lower label-change rate are better for construct-irrelevant perturbations.


In [ ]:
if "stress_summary" in report:
    stress_summary_df = (
        pd.DataFrame(report["stress_summary"])
        .T
        .reset_index()
        .rename(columns={"index": "variant"})
    )
else:
    stress_summary_df = pd.DataFrame()

stress_summary_df


In [ ]:
def plot_stress_metric(metric_name):
    if stress_summary_df.empty or metric_name not in stress_summary_df.columns:
        print(f"Missing stress metric: {metric_name}")
        return
    df = stress_summary_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"No values for stress metric: {metric_name}")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"stress_{metric_name}.png", dpi=200)
    plt.show()

for metric in [
    "invariance_label_change_rate",
    "mean_abs_invariance_score_shift",
    "mean_sensitivity_score_drop",
]:
    plot_stress_metric(metric)


## 8. Score Shift by Perturbation

For construct-irrelevant perturbations, score shifts should be close to zero. Unsupported extra claims should usually produce a score drop.


In [ ]:
stress_df = pd.DataFrame(report.get("stress_tests", []))
stress_df.head()


In [ ]:
if not stress_df.empty:
    for variant in stress_df["variant"].unique():
        sub = stress_df[stress_df["variant"] == variant]
        means = sub.groupby("perturbation")["score_shift"].mean().sort_values()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(means.index, means.values)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"Mean score shift by perturbation: {variant}")
        ax.set_ylabel("score_shift")
        ax.tick_params(axis="x", rotation=30)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"score_shift_{variant}.png", dpi=200)
        plt.show()
else:
    print("No stress_tests found. Run experiment.py with --stress.")


## 9. CALE Subscore Heatmap

This is one of the most important plots for CALE, because it shows construct-level interpretability rather than just a final label.


In [ ]:
prediction_rows = []
for pred in report.get("predictions", []):
    if pred.get("variant") != "full_cale":
        continue
    for dimension, score in pred.get("subscores", {}).items():
        prediction_rows.append({
            "id": pred["id"],
            "dimension": dimension,
            "score": score,
        })

subscore_df = pd.DataFrame(prediction_rows)
subscore_df.head()


In [ ]:
if not subscore_df.empty:
    heatmap_table = subscore_df.pivot(index="id", columns="dimension", values="score")

    fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(heatmap_table))))
    image = ax.imshow(heatmap_table.values, aspect="auto", vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label="subscore")
    ax.set_xticks(range(len(heatmap_table.columns)))
    ax.set_xticklabels(heatmap_table.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(heatmap_table.index)))
    ax.set_yticklabels(heatmap_table.index)
    ax.set_title("CALE dimension subscores")
    fig.tight_layout()
    fig.savefig(OUTDIR / "cale_subscore_heatmap.png", dpi=200)
    plt.show()
else:
    print("No full_cale subscores found.")


## 10. FEVER-Specific Optional View

If your generated dataset still contains `reference_label` from FEVER, this plot shows how evaluator outputs distribute across FEVER claim labels.


In [ ]:
rows = []
for pred in report.get("predictions", []):
    raw = pred.get("raw", {})
    # The raw report does not repeat reference_label, so we look it up from the original dataset if available.
    # If your results JSON does not include it, this section will be skipped.

if "predictions" in report and pred_df is not None and "reference_label" in pred_df.columns:
    fever_view = pred_df[["variant", "reference_label", "label"]].copy()
else:
    fever_view = pd.DataFrame()

fever_view.head() if not fever_view.empty else "No reference_label found in predictions."


## 11. Export Tables

Save metric and stress tables as CSV for paper tables.


In [ ]:
metrics_df.to_csv(OUTDIR / "metrics_table.csv", index=False)
if not stress_summary_df.empty:
    stress_summary_df.to_csv(OUTDIR / "stress_summary_table.csv", index=False)
if not subscore_df.empty:
    subscore_df.to_csv(OUTDIR / "cale_subscores_long.csv", index=False)
if not pred_df.empty:
    pred_df.to_csv(OUTDIR / "predictions_table.csv", index=False)

print(f"Saved figures and tables to: {OUTDIR.resolve()}")


## Attack-Aware Dimension Plots

These plots focus on the new attack-aware CALE dimensions.

In [ ]:
if not pred_df.empty:
    attack_dims = ["Misinformation Detection", "Framing Resistance"]
    rows = []
    for _, row in pred_df.iterrows():
        for dim in attack_dims:
            if isinstance(row.get("subscores"), dict) and dim in row["subscores"]:
                rows.append({"variant": row["variant"], "dimension": dim, "score": row["subscores"][dim]})
    attack_df = pd.DataFrame(rows)
    attack_df.head()
else:
    print("No predictions found.")


In [ ]:
if "attack_df" in globals() and not attack_df.empty:
    summary = attack_df.groupby(["variant", "dimension"]) ["score"].mean().reset_index()
    for dim in summary["dimension"].unique():
        sub = summary[summary["dimension"] == dim]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(sub["variant"], sub["score"])
        ax.set_title(dim)
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=30)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"attack_dim_{dim.replace(" ", "_")}.png", dpi=200)
        plt.show()
else:
    print("No attack-aware subscores available.")
